In this model application, we'll create a <strong> GUI that allows the user to enter NYC home characteristics and give a predicted lot price</strong>
and parameters for their home and it will spit out a predicted price.

We'll be using `gradio` for this application. 


In [1]:
# Imports
import pandas as pd
import numpy as np
import gradio as gr
import joblib
from sklearn.preprocessing import PowerTransformer

# Load the Linear Regression model.
lm = joblib.load("models/ex1_main.pkl")

In [2]:
# This converts the user's selection into a lot price prediction.
def predict(select_bldgclass_C6,select_bldgclass_D4,select_unitsres,select_unitstotal,select_numfloors,select_lotarea,select_bldgarea):


    # Infer the last three binary features based on the user's answer.
    # Doing it internally means less questions for the user and can easily
    # be inferred from the previous questions.

    # unitsres
    # If the number of residential buildings is over 5...
    # It's 1
    if select_unitsres > 5.0:
        select_highcluster_unitsres = 1 
    else:
        select_highcluster_unitsres = 0
        
    # bldgarea
    # if the sqft of the building > 5400...
    # It's 1
    if select_bldgarea > 5400:
        select_highcluster_bldgarea = 1
    else:
        select_highcluster_bldgarea = 0

    # unitstotal
    # if the number of total buildings is over 8...
    # It's 1
    if select_unitstotal > 8.0:
        select_highcluster_unitstotal = 1
    else:
        select_highcluster_unitstotal = 0


    # Converts class features to 0 and 1
    # (because they're given True, False)
    class_list = [select_bldgclass_C6, select_bldgclass_D4]

    for bldgclass in class_list:
        if bldgclass == True:
            bldgclass = 1
        else: 
            bldgclass = 0

    # First, we convert the values into a dict.
    tester_row = {
    'is_high_cluster_unitstotal' : select_highcluster_unitstotal,
    'is_high_cluster_bldgarea': select_highcluster_bldgarea ,
    'is_high_cluster_unitsres': select_highcluster_unitsres,
    'unitstotal' : select_unitstotal,
    'lotarea': select_lotarea,
    'bldgclass_D4': select_bldgclass_D4,
    'unitsres': 2,
    'bldgarea': 1100,
    'bldgclass_C6': select_bldgclass_C6,
    'numfloors' : select_numfloors,
    }

    # Then, we transform it into a dataframe.
    tester_row = pd.DataFrame([tester_row])

    # Finally, we transform the values to how the model expects it.
    pt = PowerTransformer(method='yeo-johnson')

    tester_row['unitstotal'] = np.log1p(tester_row[['unitstotal']])       # log transformation
    tester_row['lotarea'] = np.log1p(tester_row[['lotarea']])             # log transformation
    tester_row['unitsres'] = pt.fit_transform(tester_row[['unitsres']])   # yeo johnson
    tester_row['bldgarea'] = pt.fit_transform(tester_row[['bldgarea']])   # yeo johnson

    # Predict and round the result
    result = lm.predict(tester_row)[0]
    result = result.round(2)
    
    return result


In [3]:
# Checks to see if the function above works correctly.
check_1 = predict(True,False,3,9,3,1900,2400)
check_2 = predict(False,False,3,9,3,1900,2400)
check_3 = predict(False,True,3,9,3,1900,2400)
check_4 = predict(False,True,7,9,3,1900,2400)
check_5 = predict(False,True,7,11,3,1900,2400)
check_6 = predict(False,True,7,11,5,1900,2400)

# The prints. Each one of these should be different.
print(check_1)
print(check_2)
print(check_3)
print(check_4)
print(check_5)
print(check_6)


405734.39
616216.2
431623.66
432264.1
409660.05
446698.28


In [4]:
# Use Gradio Blocks to create a GUI! This should show up in the output cell.
with gr.Blocks() as house_lot_prediction:

    gr.Markdown("## NYC Home Price Predictor")
    gr.Markdown("### Enter Property Details")

    # User's GUI selections
    select_bldgclass_C6 = gr.Checkbox(label="Is your building `C6` class?")
    select_bldgclass_D4 = gr.Checkbox(label="Is your building `D4` class?")
    select_unitsres = gr.Number(label="How many residential units on your land?")
    select_unitstotal = gr.Number(label="How many total buildings on your land?")
    select_numfloors = gr.Number(label="How many floors?")
    select_lotarea = gr.Number(label="How big is your plot of land?")
    select_bldgarea = gr.Number(label="How large is the building area?")

    gr.Markdown("### Prediction")

    # Prediction output and submit button
    outputs = gr.Number(label="Home Price")
    submit = gr.Button("Predict Price")

    # Click action functionality
    submit.click(
        fn=predict,
        inputs=[
            select_bldgclass_C6,
            select_bldgclass_D4,
            select_unitsres,
            select_unitstotal,
            select_numfloors,
            select_lotarea,
            select_bldgarea,
            ],
        outputs=outputs
    )

house_lot_prediction.launch()

# Click the URL link to have it show up in your browser. (looks a little nicer)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
